In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd "/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase02"

/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase02


# Six Degrees: búsqueda en la red de actores de IMDb

**Curso:** SI3003 — Inteligencia Artificial

Este es el tercer problema de búsqueda que resuelves con la misma arquitectura de las dos clases anteriores (grafo `A–H`, laberintos). Cambia la fuente del problema — ya no es un diccionario escrito a mano ni una cuadrícula con muros, sino dos tablas CSV conectadas por una tercera — pero la pregunta de fondo es la misma: **dado un estado, ¿cuáles son sus vecinos, y qué algoritmo de frontier los recorre en el orden correcto?**

## El problema (Six Degrees of Kevin Bacon)

Según el juego "Six Degrees of Kevin Bacon", cualquier persona de la industria de Hollywood puede conectarse con Kevin Bacon en máximo seis pasos, donde cada paso es una película en la que dos actores coincidieron. Formalizado como problema de búsqueda:

- **Estados:** personas (actores).
- **Acciones:** películas — una película conecta a un actor con todos los demás actores de esa misma película.
- **Estado inicial / estado objetivo:** los dos actores que queremos conectar.
- **Costo:** cada arista vale 1 (compartir una película). No existe una heurística natural para estimar "qué tan cerca" está un actor de otro sin haber resuelto ya el grafo — por eso este problema se resuelve con **búsqueda no informada**, y específicamente BFS, porque queremos el camino con **menos películas de por medio**, no cualquier camino.

## Qué vas a implementar

1. `neighbors_for_person(person_id)` — cómo se generan los vecinos a partir de `people` y `movies` (Sección 3).
2. `shortest_path(source, target)` — BFS desde cero, con una optimización que no usamos en los notebooks anteriores: revisar si el nodo es la meta **al generarlo**, no al sacarlo de la frontier (Sección 4).

## Datos

Dos datasets con la misma estructura (`people.csv`, `movies.csv`, `stars.csv`):

- `small/`: ~15 actores, para desarrollar y depurar rápido.
- `large/`: la base completa de IMDb (~1 millón de personas), para la demostración final a escala real.

## 1. Carga de datos (entregado)

Tres diccionarios en memoria:

- `names`: nombre (en minúscula) → conjunto de `person_id` (puede haber homónimos).
- `people`: `person_id` → `{name, birth, movies}`, donde `movies` es el conjunto de `movie_id` en los que actuó.
- `movies`: `movie_id` → `{title, year, stars}`, donde `stars` es el conjunto de `person_id` que actuaron en ella.

Este es exactamente el mismo patrón que `graph` en el notebook de grafos y `Maze` en el de laberintos: una estructura que, dado un estado, permite recuperar información suficiente para generar vecinos.

In [4]:
import csv
import time
from itertools import count
from pathlib import Path

names = {}
people = {}
movies = {}


def load_data(directory):
    """Carga people.csv, movies.csv y stars.csv desde `directory` a memoria."""
    names.clear()
    people.clear()
    movies.clear()

    with open(f"{directory}/people.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            people[row["id"]] = {
                "name": row["name"],
                "birth": row["birth"],
                "movies": set()
            }
            names.setdefault(row["name"].lower(), set()).add(row["id"])

    with open(f"{directory}/movies.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            movies[row["id"]] = {
                "title": row["title"],
                "year": row["year"],
                "stars": set()
            }

    with open(f"{directory}/stars.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                people[row["person_id"]]["movies"].add(row["movie_id"])
                movies[row["movie_id"]]["stars"].add(row["person_id"])
            except KeyError:
                pass


def person_id_for_name(name):
    """Resuelve un nombre a un person_id. Si hay homónimos, devuelve la lista de ids."""
    ids = list(names.get(name.lower(), set()))
    if len(ids) == 0:
        return None
    if len(ids) == 1:
        return ids[0]
    return ids  # ambiguo: el llamador decide cuál usar

In [5]:
load_data("degrees")

print("Personas:", len(people))
print("Películas:", len(movies))
print("Kevin Bacon actuó en:", [movies[m]["title"] for m in people[person_id_for_name("Kevin Bacon")]["movies"]])

Personas: 16
Películas: 5
Kevin Bacon actuó en: ['Apollo 13', 'A Few Good Men']


## 2. Estructuras de búsqueda (entregado)

`Node`, `StackFrontier` y `QueueFrontier` son las mismas piezas de los dos notebooks anteriores, tomadas de `util.py`. Una diferencia de nomenclatura respecto al laberinto: aquí el nodo guarda `action` (el `movie_id` que te trajo hasta este estado) en vez de `cost`. No necesitamos acumular costo porque **todas las aristas valen 1** — el número de grados de separación es simplemente la longitud del camino, y BFS ya garantiza que esa longitud sea mínima sin necesidad de comparar costos como hacía UCS.

In [6]:
class Node:
    def __init__(self, state, parent, action):
        self.state = state
        self.parent = parent
        self.action = action


class StackFrontier:
    def __init__(self):
        self.frontier = []

    def add(self, node):
        self.frontier.append(node)

    def contains_state(self, state):
        return any(node.state == state for node in self.frontier)

    def empty(self):
        return len(self.frontier) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")
        node = self.frontier[-1]
        self.frontier = self.frontier[:-1]
        return node


class QueueFrontier(StackFrontier):
    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")
        node = self.frontier[0]
        self.frontier = self.frontier[1:]
        return node

## 3. `neighbors_for_person` — actividad

Esta es la traducción del problema a "vecinos". Dado un `person_id`:

1. Recupera el conjunto de películas en las que actuó (`people[person_id]["movies"]`).
2. Para cada película, recupera el conjunto de actores de esa película (`movies[movie_id]["stars"]`).
3. Devuelve el conjunto de pares `(movie_id, person_id_del_vecino)` — **nota el orden de la tupla**: aquí va primero la acción (película) y después el estado vecino, al revés que en `maze.neighbors()`, porque así lo necesita `shortest_path` para reconstruir el camino como pide el enunciado.

### Tareas

1. Complete `neighbors_for_person`.
2. Verifique con `maze1` — perdón, con Kevin Bacon — que los vecinos incluyen a sus coprotagonistas.

In [7]:
def neighbors_for_person(person_id):
    """Devuelve (movie_id, person_id) para todas las personas
    que compartieron al menos una película con person_id."""
    # TODO: implemente esta función.
    #
    # Pistas:
    #   - people[person_id]["movies"] es el conjunto de películas de esta persona.
    #   - movies[movie_id]["stars"] es el conjunto de personas de esa película.
    #   - El resultado es un set() de tuplas (movie_id, co_star_id).
    raise NotImplementedError

In [8]:
kevin_bacon_id = person_id_for_name("Kevin Bacon")
vecinos = neighbors_for_person(kevin_bacon_id)

for movie_id, co_star_id in sorted(vecinos):
    print(f"{movies[movie_id]['title']}: {people[co_star_id]['name']}")

NotImplementedError: 

In [ ]:
# Pruebas públicas para neighbors_for_person
tom_hanks_id = person_id_for_name("Tom Hanks")
vecinos_kb = {p for _, p in neighbors_for_person(kevin_bacon_id)}

assert tom_hanks_id in vecinos_kb  # coincidieron en Apollo 13

# Kevin Bacon SÍ aparece en su propio conjunto de vecinos: es una de las
# `stars` de cada película en la que actuó, y no lo estamos excluyendo
# (igual que en el degrees.py original). shortest_path lo maneja aparte
# con el caso especial source == target, así que esto no rompe nada.
assert kevin_bacon_id in vecinos_kb

print("✓ neighbors_for_person pasó las pruebas.")

## 4. `shortest_path` — actividad principal

Implemente BFS desde `source` hasta `target`. Debe devolver una lista de tuplas `(movie_id, person_id)` — el camino, no el nodo — o `None` si no hay conexión.

### Tareas

1. Caso especial: si `source == target`, el camino es `[]` (cero grados de separación).
2. Use `QueueFrontier` (BFS, no DFS/UCS — no hay costo que acumular ni heurística que priorizar).
3. **Optimización pedida en el enunciado:** revise si un vecino es la meta **al generarlo** (antes de agregarlo a la frontier), no al sacarlo de la frontier como hacíamos en los notebooks anteriores. Con ~1 millón de personas en `large`, evitar encolar nodos innecesarios importa.
4. Al encontrar la meta, reconstruya el camino siguiendo `node.parent` hacia atrás (igual que `reconstruct_path`, pero aquí cada paso guarda `(action, state)` en vez de solo `state`).

In [ ]:
def shortest_path(source, target):
    """
    BFS desde source hasta target sobre la red de actores.
    Devuelve una lista de (movie_id, person_id), o None si no hay camino.
    """
    # TODO: implemente BFS.
    #
    # Pistas:
    #   - Caso especial: si source == target, el camino es [] (0 grados).
    #   - Use QueueFrontier (BFS), no StackFrontier ni una cola de prioridad.
    #   - Revise si un vecino es la meta AL GENERARLO (antes de agregarlo a
    #     la frontier), no al sacarlo de la frontier. Con ~1M de personas en
    #     `large`, evitar encolar nodos de más importa para el desempeño.
    #   - Para reconstruir el camino, siga node.parent hacia atrás guardando
    #     (node.action, node.state) en cada paso, y luego invierta la lista.
    raise NotImplementedError

In [ ]:
def describe_path(source, path):
    """Imprime el camino en el mismo formato que main() en degrees.py."""
    current = source
    for i, (movie_id, person_id) in enumerate(path, start=1):
        p1 = people[current]["name"]
        p2 = people[person_id]["name"]
        title = movies[movie_id]["title"]
        print(f"{i}: {p1} y {p2} actuaron juntos en {title}")
        current = person_id


kb = person_id_for_name("Kevin Bacon")
th = person_id_for_name("Tom Hanks")

path = shortest_path(kb, th)
print(f"{len(path)} grados de separación.")
describe_path(kb, path)

In [ ]:
# Pruebas públicas para shortest_path (dataset small)
jn = person_id_for_name("Jack Nicholson")
bp = person_id_for_name("Bill Paxton")

path_kb_th = shortest_path(kb, th)
path_jn_bp = shortest_path(jn, bp)

assert len(path_kb_th) == 1
assert len(path_jn_bp) == 2

print("✓ shortest_path pasó las pruebas públicas.")

## 5. Demostración a escala real (`large`)

El dataset `large` tiene ~1 millón de personas y ~344 mil películas. Cargarlo toma unos segundos (es E/S de disco, no búsqueda); una vez en memoria, BFS encuentra el camino en fracciones de segundo, incluso sobre un grafo de este tamaño — la parte cara es leer los CSV, no recorrer el grafo.

> Ejecuta esta sección solo si tienes los archivos `large/people.csv`, `large/movies.csv` y `large/stars.csv` disponibles (no se incluyen en este repositorio por su tamaño).

In [ ]:
t0 = time.time()
load_data("large")
print(f"Cargado en {time.time() - t0:.1f} s — {len(people)} personas, {len(movies)} películas.")

In [ ]:
ew_id = person_id_for_name("Emma Watson")
jl_id = person_id_for_name("Jennifer Lawrence")

t0 = time.time()
path_large = shortest_path(ew_id, jl_id)
elapsed = time.time() - t0

print(f"BFS tomó {elapsed:.3f} s sobre ~1M de personas.")
assert path_is_valid(path_large, ew_id, jl_id)

print(f"{len(path_large)} grados de separación.")
describe_path(ew_id, path_large)

Nota: el enunciado original documenta un camino distinto (vía Brendan Gleeson y Michael Fassbender) para el mismo par y también de 3 grados — es exactamente el caso que anticipa la especificación: *"si hay múltiples caminos de longitud mínima, tu función puede devolver cualquiera de ellos"*. Lo que sí es invariante, y lo que deberíamos verificar, es el **número de grados**, no la ruta específica.

## 6. Preguntas de cierre

1. ¿Por qué BFS y no DFS? ¿Qué pasaría si usaras DFS sobre `large` para encontrar *algún* camino entre dos actores lejanos?
2. ¿Por qué no tiene sentido plantear este problema con A* o Greedy Best-First Search? ¿Qué tendría que existir en los datos para que sí tuviera sentido?
3. La optimización de revisar la meta al generar el nodo (no al sacarlo de la frontier) cambia el *momento* de la comprobación, pero no el algoritmo. ¿Por qué de todas formas mejora el desempeño en un grafo de 1 millón de nodos?
4. `neighbors_for_person` construye el conjunto de vecinos completo cada vez que se llama, incluso si ya se calculó antes para ese mismo `person_id`. ¿Qué estructura de datos usarías para no repetir ese trabajo si tuvieras que correr `shortest_path` muchas veces sobre el mismo grafo?
5. Compara este problema con el laberinto: allá el estado (posición) determinaba los vecinos por geometría; aquí los determina una relación en una tabla. ¿Qué tienen en común las tres representaciones de vecinos que has escrito este curso (diccionario, `Maze.neighbors`, `neighbors_for_person`)?